# Feature Engineering – Olist E-Commerce

Xây dựng 10 biến đặc trưng theo **Bảng 2.2**, chuẩn hóa và xuất dataset cho mô hình.

**Luồng xử lý:**
1. Import & Đọc dữ liệu  
2. Tiền xử lý (datetime, lọc đơn đã giao)  
3. Tính biến nhóm Giao hàng (`delivery_delay`, `lead_time`, `seller_speed`)  
4. Tính biến nhóm Giao dịch (`freight_ratio`, `installments`, `price`)  
5. Tính biến nhóm Sản phẩm (`desc_length`, `photos_qty`)  
6. Tính biến nhóm Lịch sử (`seller_rep`, `customer_order_count`)  
7. Merge tất cả + gắn nhãn (`review_score`, `review_binary`)  
8. Xử lý ngoại lệ (Clipping)  
9. Chuẩn hóa Min-Max toàn bộ 10 biến  
10. Kiểm tra & Xuất file  
11. Tách tập Train/Test  
12. Mô hình phân loại (Random Forest, XGBoost, MLP)

## Bước 1 – Import thư viện & Đọc dữ liệu

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# Đọc các file csv (đảm bảo file nằm cùng thư mục code)
orders   = pd.read_csv('olist_orders_dataset_clean.csv')
items    = pd.read_csv('olist_order_items_dataset_clean.csv')
payments = pd.read_csv('olist_order_payments_dataset_clean.csv')
products = pd.read_csv('olist_products_dataset_clean.csv')
reviews  = pd.read_csv('olist_order_reviews_dataset_clean.csv')
customers = pd.read_csv('olist_customers_dataset_clean.csv')

print('✓ Đọc dữ liệu hoàn tất')
print(f'  orders:    {orders.shape}')
print(f'  items:     {items.shape}')
print(f'  payments:  {payments.shape}')
print(f'  products:  {products.shape}')
print(f'  reviews:   {reviews.shape}')
print(f'  customers: {customers.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'olist_orders_dataset_clean.csv'

## Bước 2 – Tiền xử lý: Chuyển đổi datetime & Lọc đơn đã giao

In [ ]:
# Chuyển đổi các cột thời gian
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    # errors='coerce' sẽ biến các giá trị lỗi như "0" thành NaT
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Lọc đơn hàng đã giao (để đảm bảo có đủ ngày nhận hàng)
df_master = orders[orders['order_status'] == 'delivered'].copy()

print(f'✓ Tổng đơn đã giao: {len(df_master):,}')

## Bước 3 – Nhóm Giao hàng: `delivery_delay`, `lead_time`, `seller_speed`

In [ ]:
# delivery_delay: Độ trễ giao hàng (ngày)
# Dương = giao TRỄ hơn ước tính, Âm = giao SỚM hơn ước tính
df_master['raw_delivery_delay'] = (
    df_master['order_delivered_customer_date'] - df_master['order_estimated_delivery_date']
).dt.total_seconds() / 86400

# lead_time: Tổng thời gian từ đặt hàng đến nhận hàng (ngày)
df_master['raw_lead_time'] = (
    df_master['order_delivered_customer_date'] - df_master['order_purchase_timestamp']
).dt.total_seconds() / 86400

# seller_speed: Thời gian người bán chuẩn bị và bàn giao vận chuyển (ngày)
df_master['raw_seller_speed'] = (
    df_master['order_delivered_carrier_date'] - df_master['order_approved_at']
).dt.total_seconds() / 86400

print('✓ Nhóm Giao hàng:')
print(df_master[['raw_delivery_delay', 'raw_lead_time', 'raw_seller_speed']].describe().round(2))

## Bước 4 – Nhóm Giao dịch: `freight_ratio`, `installments`, `price`

In [ ]:
# freight_ratio: Tỷ lệ phí vận chuyển so với giá trị hàng hóa
item_agg = items.groupby('order_id').agg(
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum')
).reset_index()
item_agg['raw_freight_ratio'] = item_agg['freight_value'] / (item_agg['price'] + 0.001)

# raw_price: Tổng giá trị hàng hóa (BRL)
item_agg['raw_price'] = item_agg['price']

# installments: Số kỳ trả góp lớn nhất trong đơn hàng
pay_agg = payments.groupby('order_id').agg(
    raw_installments=('payment_installments', 'max')
).reset_index()

print('✓ Nhóm Giao dịch:')
print(item_agg[['raw_freight_ratio', 'raw_price']].describe().round(2))

## Bước 5 – Nhóm Sản phẩm: `desc_length`, `photos_qty`

In [ ]:
# Gắn thông tin sản phẩm vào order items
order_items_prod = items.merge(products, on='product_id')

# desc_length: Độ dài mô tả sản phẩm trung bình trong đơn (ký tự)
# photos_qty : Số lượng ảnh sản phẩm trung bình trong đơn
prod_agg = order_items_prod.groupby('order_id').agg(
    raw_desc_length=('product_description_lenght', 'mean'),
    raw_photos_qty=('product_photos_qty', 'mean')
).reset_index()

print('✓ Nhóm Sản phẩm:')
print(prod_agg[['raw_desc_length', 'raw_photos_qty']].describe().round(2))

## Bước 6 – Nhóm Lịch sử: `seller_rep`, `customer_order_count`

In [ ]:
# Tạo bảng tạm gộp thông tin cần thiết
df_history = (
    orders[['order_id', 'customer_id', 'order_purchase_timestamp']]
    .merge(items[['order_id', 'seller_id']], on='order_id', how='inner')
    .merge(reviews[['order_id', 'review_score', 'review_creation_date']], on='order_id', how='inner')
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
)

# Đảm bảo định dạng datetime
df_history['order_purchase_timestamp'] = pd.to_datetime(
    df_history['order_purchase_timestamp'], errors='coerce')
df_history['review_creation_date'] = pd.to_datetime(
    df_history['review_creation_date'], errors='coerce')

# ── seller_rep: Danh tiếng người bán (điểm TB tích lũy, tránh data leakage) ──
# Sắp xếp theo ngày tạo đánh giá để tính điểm tích lũy từ quá khứ → hiện tại
df_history = df_history.sort_values('review_creation_date')
df_history['raw_seller_rep'] = (
    df_history.groupby('seller_id')['review_score']
    .expanding()
    .mean()
    .shift(1)  # Shift 1 để không tính điểm của chính đơn hàng hiện tại
    .reset_index(level=0, drop=True)
)
# Đơn đầu tiên của seller (NaN do shift) → điền giá trị trung bình mặc định
df_history['raw_seller_rep'] = df_history['raw_seller_rep'].fillna(4.0)

# ── customer_order_count: Số lần mua tích lũy của khách hàng ──────────────────
# Sắp xếp theo ngày mua để cumcount đúng thứ tự thời gian
df_history = df_history.sort_values('order_purchase_timestamp')
df_history['raw_customer_order_count'] = (
    df_history.groupby('customer_unique_id').cumcount()  # 0-indexed: lần đầu = 0
)

# Lọc bỏ dòng trùng order_id (1 đơn có thể nhiều items)
history_agg = df_history.drop_duplicates(subset=['order_id'], keep='last')

print('✓ Nhóm Lịch sử:')
print(history_agg[['raw_seller_rep', 'raw_customer_order_count']].describe().round(3))

## Bước 7 – Merge tất cả & Gắn nhãn (`review_score`, `review_binary`)

In [ ]:
# Bắt đầu từ df_master (đơn đã giao) với 3 biến giao hàng
df_final = df_master[['order_id', 'raw_delivery_delay', 'raw_lead_time', 'raw_seller_speed']].copy()

# Merge nhóm Giao dịch
df_final = df_final.merge(
    item_agg[['order_id', 'raw_freight_ratio', 'raw_price']], on='order_id', how='left')
df_final = df_final.merge(pay_agg, on='order_id', how='left')

# Merge nhóm Sản phẩm
df_final = df_final.merge(prod_agg, on='order_id', how='left')

# Merge nhóm Lịch sử
df_final = df_final.merge(
    history_agg[['order_id', 'raw_seller_rep', 'raw_customer_order_count']],
    on='order_id', how='left')

# Gắn nhãn review_score (điểm đánh giá gốc 1–5)
review_scores = reviews.groupby('order_id')['review_score'].mean().reset_index()
df_final = df_final.merge(review_scores, on='order_id', how='inner')

# Tạo nhãn nhị phân: 0 = Không hài lòng (≤3), 1 = Hài lòng (>3)
df_final['review_binary'] = df_final['review_score'].apply(lambda x: 0 if x <= 3 else 1)

print(f'✓ Sau merge: {df_final.shape}')
print(f'  Phân phối review_binary:')
print(df_final['review_binary'].value_counts())

## Bước 8 – Xử lý ngoại lệ (Clipping) & Điền giá trị thiếu

In [ ]:
# Clipping để thang đo 0-1 không bị méo do outlier cực đoan
df_final['raw_delivery_delay'] = df_final['raw_delivery_delay'].clip(-30, 30)
# Trễ/Sớm quá 30 ngày coi như mức trần

df_final['raw_freight_ratio']  = df_final['raw_freight_ratio'].clip(0, 2)
# Phí ship đắt gấp 2 lần giá trị hàng là tối đa

df_final['raw_price']          = df_final['raw_price'].clip(0, 2000)
# Giới hạn 2000 BRL để đơn hàng cực đắt không làm lệch thang đo

# Điền giá trị thiếu bằng 0 (các biến seller_speed, desc_length, photos_qty
# có thể thiếu với một số đơn hàng không đủ thông tin)
raw_cols = [c for c in df_final.columns if c.startswith('raw_')]
df_final[raw_cols] = df_final[raw_cols].fillna(0)

print('✓ Clipping & Fillna hoàn tất')
print(f'  Số dòng: {len(df_final):,}')
print(f'  Missing sau fillna: {df_final[raw_cols].isna().sum().sum()}')

## Bước 9 – Chuẩn hóa Min-Max toàn bộ 10 biến

In [ ]:
# Chuẩn hóa tất cả biến raw_ về thang đo [0, 1] cùng một lần
scaler = MinMaxScaler()
norm_cols = [c.replace('raw_', 'norm_') for c in raw_cols]

df_final[norm_cols] = scaler.fit_transform(df_final[raw_cols])

print('✓ Chuẩn hóa Min-Max hoàn tất')
print(f'  Biến norm: {norm_cols}')
print(f'  Kiểm tra min/max sau chuẩn hóa:')
print(df_final[norm_cols].agg(['min', 'max']).round(4))

## Bước 10 – Kiểm tra & Xuất file

In [ ]:
# Xem trước dataset cuối cùng
print(f'Dataset cuối cùng: {df_final.shape}')
print(f'Các cột: {list(df_final.columns)}')
df_final.head(3)

In [ ]:
# Xuất file – dùng cho EDA và mô hình
df_final.to_csv('olist_final_dataset_labeled.csv', index=False)
print('✓ Đã lưu: olist_final_dataset_labeled.csv')

## Bước 11 – Tách tập Train/Test (80/20, stratified)

In [ ]:
# X: tất cả biến norm_; y: nhãn nhị phân
feature_cols = [c for c in df_final.columns if c.startswith('norm_')]
X = df_final[feature_cols]
y = df_final['review_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Giữ tỷ lệ lớp trong cả train và test
)

print(f'✓ Tách Train/Test hoàn tất')
print(f'  X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'  Phân phối y_train: {y_train.value_counts().to_dict()}')
print(f'  Phân phối y_test:  {y_test.value_counts().to_dict()}')

count_class_0 = (y_train == 0).sum()
count_class_1 = (y_train == 1).sum()
scale_pos_weight = count_class_0 / count_class_1
print(f'\n  scale_pos_weight (XGBoost): {scale_pos_weight:.4f}')

## Bước 12 – Mô hình phân loại

### 12.1 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Báo cáo phân loại của Random Forest trên tập Test:\n')
print(classification_report(
    y_test, y_pred_rf,
    target_names=['Không hài lòng', 'Hài lòng']
))

### 12.2 XGBoost (không xử lý mất cân bằng)

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print('Báo cáo phân loại của XGBoost trên tập Test:\n')
print(classification_report(
    y_test, y_pred_xgb,
    target_names=['Không hài lòng', 'Hài lòng']
))

### 12.3 XGBoost (có xử lý mất cân bằng – scale_pos_weight)

In [ ]:
print(f'Số lượng class 0 (Không hài lòng) trong tập train: {count_class_0}')
print(f'Số lượng class 1 (Hài lòng) trong tập train: {count_class_1}')
print(f'Giá trị scale_pos_weight được áp dụng: {scale_pos_weight:.4f}\n')

xgb_balanced = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,  # Xử lý mất cân bằng lớp
    eval_metric='logloss',
    random_state=42
)
xgb_balanced.fit(X_train, y_train)
y_pred_xgb_bal = xgb_balanced.predict(X_test)

print('Báo cáo phân loại của XGBoost (Đã xử lý Imbalanced):\n')
print(classification_report(
    y_test, y_pred_xgb_bal,
    target_names=['Không hài lòng', 'Hài lòng']
))

### 12.4 MLP (Neural Network)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    random_state=42
)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)

print('Báo cáo phân loại của MLP trên tập Test:\n')
print(classification_report(
    y_test, y_pred_mlp,
    target_names=['Không hài lòng', 'Hài lòng'],
    digits=3
))